# Exploratory Analysis of Geolocational Data
Pipeline: **load → clean → summarise → map → density → cluster → export.**

Set `DATA_PATH` in the config cell to your CSV (needs latitude/longitude columns; a timestamp column is optional).
If the file isn't found, a synthetic dataset is generated so the whole notebook still runs end to end.

In [ ]:
# Install once if needed:  pip install pandas numpy matplotlib seaborn scikit-learn folium
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap, MarkerCluster
from sklearn.cluster import DBSCAN
from pathlib import Path

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.5f}".format)

## 1. Config

In [ ]:
DATA_PATH = "locations.csv"      # <- your file
LAT_COL, LON_COL, TIME_COL = "latitude", "longitude", "timestamp"   # TIME_COL may be absent
EPS_KM, MIN_SAMPLES = 0.5, 15    # DBSCAN: neighbourhood radius (km) and min points
OUT_DIR = Path("outputs"); OUT_DIR.mkdir(exist_ok=True)

## 2. Load data (falls back to synthetic data)

In [ ]:
def make_synthetic(n=5000, seed=42):
    rng = np.random.default_rng(seed)
    # Hotspots around Kolhapur, Pune, Mumbai-ish offsets + background noise
    centers = [(16.705, 74.243), (16.70, 74.26), (16.68, 74.22), (16.73, 74.20)]
    parts = []
    for lat, lon in centers:
        k = n // 6
        parts.append(pd.DataFrame({
            "latitude": rng.normal(lat, 0.008, k),
            "longitude": rng.normal(lon, 0.008, k)}))
    noise = pd.DataFrame({
        "latitude": rng.uniform(16.6, 16.8, n // 5),
        "longitude": rng.uniform(74.1, 74.35, n // 5)})
    df = pd.concat(parts + [noise], ignore_index=True)
    df["timestamp"] = pd.Timestamp("2026-01-01") + pd.to_timedelta(
        rng.integers(0, 60 * 24 * 90, len(df)), unit="m")
    df.loc[rng.choice(len(df), 25, replace=False), "latitude"] = np.nan   # dirty rows
    df.loc[rng.choice(len(df), 10, replace=False), "longitude"] = 999     # invalid coords
    return pd.concat([df, df.sample(50, random_state=1)], ignore_index=True)  # duplicates

if Path(DATA_PATH).exists():
    raw = pd.read_csv(DATA_PATH)
    print(f"Loaded {DATA_PATH}")
else:
    raw = make_synthetic()
    LAT_COL, LON_COL, TIME_COL = "latitude", "longitude", "timestamp"
    print("No data file found -> using synthetic data")

print(raw.shape); raw.head()

## 3. Clean
Drop nulls, invalid coordinates, exact duplicates; parse timestamps.

In [ ]:
df = raw.copy()
n0 = len(df)
df[LAT_COL] = pd.to_numeric(df[LAT_COL], errors="coerce")
df[LON_COL] = pd.to_numeric(df[LON_COL], errors="coerce")
df = df.dropna(subset=[LAT_COL, LON_COL])
n_null = n0 - len(df)
df = df[df[LAT_COL].between(-90, 90) & df[LON_COL].between(-180, 180)]
n_invalid = n0 - n_null - len(df)
before = len(df); df = df.drop_duplicates(); n_dup = before - len(df)

has_time = TIME_COL in df.columns
if has_time:
    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.rename(columns={LAT_COL: "lat", LON_COL: "lon"}).reset_index(drop=True)

print(f"Start {n0} | null {n_null} | invalid {n_invalid} | duplicates {n_dup} | kept {len(df)}")

## 4. Summary statistics

In [ ]:
display(df[["lat", "lon"]].describe())
bbox = dict(lat_min=df.lat.min(), lat_max=df.lat.max(), lon_min=df.lon.min(), lon_max=df.lon.max())
print("Bounding box:", {k: round(v, 4) for k, v in bbox.items()})
print("Centroid:", round(df.lat.mean(), 5), round(df.lon.mean(), 5))

## 5. Spatial scatter & distributions

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 5))
ax[0].scatter(df.lon, df.lat, s=3, alpha=0.3); ax[0].set(title="Points", xlabel="lon", ylabel="lat")
sns.histplot(df.lat, bins=50, ax=ax[1]); ax[1].set_title("Latitude distribution")
sns.histplot(df.lon, bins=50, ax=ax[2]); ax[2].set_title("Longitude distribution")
plt.tight_layout(); plt.show()

## 6. Density (hexbin)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
hb = ax.hexbin(df.lon, df.lat, gridsize=50, cmap="magma", mincnt=1)
fig.colorbar(hb, label="points per cell"); ax.set(title="Point density", xlabel="lon", ylabel="lat")
plt.show()

## 7. Temporal patterns (if timestamps exist)

In [ ]:
if has_time and df[TIME_COL].notna().any():
    t = df.dropna(subset=[TIME_COL]).copy()
    t["hour"], t["dow"] = t[TIME_COL].dt.hour, t[TIME_COL].dt.day_name()
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))
    sns.countplot(x="hour", data=t, ax=ax[0], color="steelblue"); ax[0].set_title("By hour")
    order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
    sns.countplot(x="dow", data=t, order=order, ax=ax[1], color="teal"); ax[1].set_title("By weekday")
    ax[1].tick_params(axis="x", rotation=45); plt.tight_layout(); plt.show()
else:
    print("No usable timestamp column - skipping.")

## 8. Clustering with DBSCAN (haversine)
Coordinates go in as radians; `eps` is converted from km to radians.

In [ ]:
EARTH_KM = 6371.0088
coords = np.radians(df[["lat", "lon"]].to_numpy())
db = DBSCAN(eps=EPS_KM / EARTH_KM, min_samples=MIN_SAMPLES,
            metric="haversine", algorithm="ball_tree").fit(coords)
df["cluster"] = db.labels_

n_clusters = len(set(db.labels_)) - (1 if -1 in db.labels_ else 0)
print(f"Clusters: {n_clusters} | noise points: {(df.cluster == -1).sum()} ({(df.cluster == -1).mean():.1%})")

summary = (df[df.cluster >= 0].groupby("cluster")
           .agg(points=("lat", "size"), lat=("lat", "mean"), lon=("lon", "mean"))
           .sort_values("points", ascending=False))
summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
noise = df[df.cluster == -1]; ax.scatter(noise.lon, noise.lat, s=3, c="lightgrey", label="noise")
cl = df[df.cluster >= 0]
sc = ax.scatter(cl.lon, cl.lat, s=5, c=cl.cluster, cmap="tab20")
ax.scatter(summary.lon, summary.lat, marker="x", c="black", s=80, label="centroid")
ax.set(title="DBSCAN clusters", xlabel="lon", ylabel="lat"); ax.legend(); plt.show()

## 9. Interactive map (saved as HTML)

In [ ]:
sample = df.sample(min(len(df), 5000), random_state=0)
m = folium.Map(location=[df.lat.mean(), df.lon.mean()], zoom_start=12, tiles="OpenStreetMap")
HeatMap(sample[["lat", "lon"]].values.tolist(), radius=10, blur=14, name="Heatmap").add_to(m)
for cid, r in summary.iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=6 + r.points ** 0.5 / 3, color="red", fill=True,
                        tooltip=f"Cluster {cid}: {int(r.points)} points").add_to(m)
folium.LayerControl().add_to(m)
m.save(OUT_DIR / "map.html"); m

## 10. Export

In [ ]:
df.to_csv(OUT_DIR / "clean_with_clusters.csv", index=False)
summary.to_csv(OUT_DIR / "cluster_summary.csv")
print("Saved to", OUT_DIR.resolve(), "->", sorted(p.name for p in OUT_DIR.iterdir()))

## Next steps
- Tune `EPS_KM` / `MIN_SAMPLES` (try a k-distance plot)
- Add `geopandas` for shapefiles / spatial joins (wards, districts)
- Trajectory analysis (speed, dwell time) if you have per-user GPS traces